In [56]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [15]:
merchant = pd.read_parquet('/Users/sapuantalaspay/vs_projects/data/data/raw/merchants_reference.parquet', engine="fastparquet")
business = pd.read_parquet('/Users/sapuantalaspay/vs_projects/data/data/raw/business_cards_MDQ.parquet', engine="fastparquet")
consumer = pd.read_parquet('/Users/sapuantalaspay/vs_projects/data/data/raw/consumer_cards_MDQ.parquet', engine="fastparquet")

In [16]:
merchant.head()

,merchant_id,merchant_name,mcc,merchant_country,recurring_capable
0,MER_000000,Google Ads,7311,Ireland,True
1,MER_000001,Meta Ads,7311,Ireland,True
2,MER_000002,TikTok Ads,7311,Singapore,True
3,MER_000003,Yandex Direct,7311,Russia,True
4,MER_000004,LinkedIn Ads,7311,Ireland,True


In [17]:
business.head()

,transaction_date,transaction_timestamp,transaction_amount_kzt,mcc,merchant_id,channel,bank_name,country,card_number,card_tier,tokenized,is_recurring
0,1759276800000000000,2025-10-01 00:00:00,180976,7372,MER_000007,online,Kaspi,US,5228592291438845,Business,False,True
1,1759276800000000000,2025-10-01 00:00:00,153206,7372,MER_000006,online,Home Credit Bank,US,5201495142193372,Business,False,True
2,1759276800000000000,2025-10-01 00:00:00,197106,7372,MER_000007,online,Home Credit Bank,US,5201492177677288,Business,False,True
3,1759276800000000000,2025-10-01 00:01:00,189598,7372,MER_000008,online,Kaspi,US,5176513443697635,Business,False,True
4,1759276800000000000,2025-10-01 00:03:00,700571,7311,MER_000003,online,Halyk,Russia,5100611967455520,Business,False,True


In [18]:
consumer.head()

,transaction_date,transaction_timestamp,transaction_amount_kzt,mcc,merchant_id,channel,bank_name,country,card_number,card_tier,tokenized,is_recurring
0,1759276800000000000,2025-10-01 00:04:00,4788,4814,MER_000064,online,Alatau City Bank,Kazakhstan,5263907968824596,Standard,False,True
1,1759276800000000000,2025-10-01 00:10:00,5240,4814,MER_000063,online,Bank RBK,Kazakhstan,5119023663984986,Standard,False,True
2,1759276800000000000,2025-10-01 00:12:00,4576,4814,MER_000066,online,Kaspi,Kazakhstan,5228590878155154,Standard,False,True
3,1759276800000000000,2025-10-01 00:37:00,6078,4814,MER_000063,online,Home Credit Bank,Kazakhstan,5338472125333693,Standard,False,True
4,1759276800000000000,2025-10-01 00:37:00,6042,4814,MER_000065,online,Kaspi,Kazakhstan,5531514712394557,Affluent,False,True


In [39]:
business.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2997593 entries, 0 to 2997592
Data columns (total 12 columns):
 #   Column                  Dtype        
---  ------                  -----        
 0   transaction_date        object       
 1   transaction_timestamp   datetime64[s]
 2   transaction_amount_kzt  int64        
 3   mcc                     object       
 4   merchant_id             object       
 5   channel                 object       
 6   bank_name               object       
 7   country                 object       
 8   card_number             object       
 9   card_tier               object       
 10  tokenized               bool         
 11  is_recurring            bool         
dtypes: bool(2), datetime64[s](1), int64(1), object(8)
memory usage: 234.4+ MB


In [40]:
consumer.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9832487 entries, 0 to 9832486
Data columns (total 12 columns):
 #   Column                  Dtype         
---  ------                  -----         
 0   transaction_date        object        
 1   transaction_timestamp   datetime64[ms]
 2   transaction_amount_kzt  int64         
 3   mcc                     object        
 4   merchant_id             object        
 5   channel                 object        
 6   bank_name               object        
 7   country                 object        
 8   card_number             object        
 9   card_tier               object        
 10  tokenized               bool          
 11  is_recurring            bool          
dtypes: bool(2), datetime64[ms](1), int64(1), object(8)
memory usage: 768.9+ MB


In [41]:
merchant.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2165 entries, 0 to 2164
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   merchant_id        2165 non-null   object
 1   merchant_name      2165 non-null   object
 2   mcc                2165 non-null   object
 3   merchant_country   2165 non-null   object
 4   recurring_capable  2165 non-null   bool  
dtypes: bool(1), object(4)
memory usage: 69.9+ KB


## Проверка на пропущенные значения

In [ ]:
print(consumer.isnull().sum())

transaction_date          0
transaction_timestamp     0
transaction_amount_kzt    0
mcc                       0
merchant_id               0
channel                   0
bank_name                 0
country                   0
card_number               0
card_tier                 0
tokenized                 0
is_recurring              0
dtype: int64 



In [54]:
print(business.isnull().sum())

transaction_date          0
transaction_timestamp     0
transaction_amount_kzt    0
mcc                       0
merchant_id               0
channel                   0
bank_name                 0
country                   0
card_number               0
card_tier                 0
tokenized                 0
is_recurring              0
dtype: int64


In [55]:
print(merchant.isnull().sum())

merchant_id          0
merchant_name        0
mcc                  0
merchant_country     0
recurring_capable    0
dtype: int64


## Проверка бизнес смысла merchant_id и mcc 

In [31]:
multi_mcc_merchants = (
    business.groupby('merchant_id')
    .agg(
        unique_mcc=('mcc', 'nunique'),
        tx_count=('mcc', 'size')
    )
    .query('unique_mcc > 1')
    .sort_values(['unique_mcc', 'tx_count'], ascending=False)
)

multi_mcc_merchants.head(20)

,unique_mcc,tx_count
merchant_id,,
MER_000000,2,63450


In [32]:
business[business['merchant_id'] == 'MER_000000']['mcc'].value_counts(normalize=False)

mcc
7311    45749
7012    17701
Name: count, dtype: int64

In [33]:
merchant[merchant['merchant_id'] == 'MER_000000']

,merchant_id,merchant_name,mcc,merchant_country,recurring_capable
0,MER_000000,Google Ads,7311,Ireland,True


### MER_000000 имеет 2 mcc в business
дальше проверим в consumer

In [35]:
consumer_multi_mcc_merchants = (
    consumer.groupby('merchant_id')
    .agg(
        unique_mcc=('mcc', 'nunique'),
        tx_count=('mcc', 'size')
    )
    .query('unique_mcc > 1')
    .sort_values(['unique_mcc', 'tx_count'], ascending=False)
)

consumer_multi_mcc_merchants.head(20)

,unique_mcc,tx_count
merchant_id,,
MER_000000,2,87019


In [37]:
consumer[consumer['merchant_id'] == 'MER_000000']['mcc'].value_counts()

mcc
7311    43869
7012    43150
Name: count, dtype: int64

In [43]:
print("Share of MER_000000 in business:")

print(business['merchant_id'].eq('MER_000000').mean() * 100)

print()
print("Share of MER_000000 in consumer:")

print(consumer['merchant_id'].eq('MER_000000').mean() * 100)

Share of MER_000000 in business:
2.116698297600775

Share of MER_000000 in consumer:
0.8850151543551494


In [48]:
for name, df in [('business', business), ('consumer', consumer)]:
    total_rows = len(df)
    unknown_rows = df['merchant_id'].eq('MER_000000').sum()
    share = unknown_rows / total_rows * 100

    print(f'{name}:')
    print(f'total rows: {total_rows:,}')
    print(f'MER_000000 rows: {unknown_rows:,}')
    print(f'share: {share:.4f}%\n')


business:
total rows: 2,997,593
MER_000000 rows: 63,450
share: 2.1167%

consumer:
total rows: 9,832,487
MER_000000 rows: 87,019
share: 0.8850%



## salam